# 4-3. 학습 최적화와 생성 결과 분석

> **이 노트북에서 답할 질문**
> (1) 학습 루프를 실무처럼 짜려면 무엇이 더 필요한가?
> (2) 학습된 모델에게 글을 쓰게 하면, 그 결과를 어떻게 읽어야 하는가?

## 1. 지금까지의 학습 루프에 빠진 것들

4-1에서 이렇게 썼습니다.

```python
for step in range(steps):
    loss = F.cross_entropy(model(Xtr), Ytr)   # 전체 데이터를 한 번에!
    opt.zero_grad(); loss.backward(); opt.step()
```

작은 데이터라 통했지만, 실무에서는 세 가지가 빠져 있습니다.

1. **미니배치** — 데이터 전체가 메모리에 안 들어갑니다
2. **에폭/셔플** — 매번 같은 순서로 보면 학습이 편향됩니다
3. **체크포인트 & 조기 종료** — valid 최저점의 모델을 건져야 합니다

하나씩 채워 넣겠습니다.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "common.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import math, time, random
import torch
import torch.nn as nn
import torch.nn.functional as F
from common import load_sentences, set_seed, get_device, count_params, human

set_seed(1234)
device = get_device()

In [ ]:
def tokenize(s):
    return s.replace(".", " .").split()

BOS, UNK = "<s>", "<unk>"
sents = load_sentences()
sh = sents[:]
random.Random(0).shuffle(sh)
cut = int(len(sh) * 0.8)
train_sents, val_sents = sh[:cut], sh[cut:]

vocab = [BOS, UNK] + sorted({t for s in train_sents for t in tokenize(s)})
stoi = {w: i for i, w in enumerate(vocab)}
itos = {i: w for w, i in stoi.items()}
V = len(vocab)

CTX = 3
def make_dataset(ss):
    xs, ys = [], []
    for s in ss:
        ids = [stoi.get(t, stoi[UNK]) for t in [BOS] * CTX + tokenize(s)]
        for i in range(CTX, len(ids)):
            xs.append(ids[i - CTX:i]); ys.append(ids[i])
    return torch.tensor(xs), torch.tensor(ys)

Xtr, Ytr = make_dataset(train_sents)
Xva, Yva = make_dataset(val_sents)
print(f"V={V}, 학습 {len(Xtr)} / 평가 {len(Xva)} 샘플")

## 2. 미니배치 학습 루프

`TensorDataset` + `DataLoader` 로 배치를 만듭니다.
`shuffle=True` 가 매 에폭마다 순서를 섞어 줍니다.

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

BATCH = 64
train_loader = DataLoader(TensorDataset(Xtr, Ytr), batch_size=BATCH, shuffle=True)
val_loader   = DataLoader(TensorDataset(Xva, Yva), batch_size=BATCH)

print(f"배치 크기 {BATCH},  에폭당 {len(train_loader)} 스텝")
xb, yb = next(iter(train_loader))
print(f"배치 하나: X{tuple(xb.shape)}  Y{tuple(yb.shape)}")

In [ ]:
class NPLM(nn.Module):
    def __init__(self, V, d=64, hidden=128, ctx=CTX, dropout=0.2):
        super().__init__()
        self.emb = nn.Embedding(V, d)
        self.drop = nn.Dropout(dropout)          # 과적합 완화 장치
        self.H = nn.Linear(ctx * d, hidden)
        self.U = nn.Linear(hidden, V)

    def forward(self, idx):
        x = self.drop(self.emb(idx).flatten(1))
        return self.U(torch.tanh(self.H(x)))

@torch.no_grad()
def eval_loss(model, loader):
    model.eval()
    total, n = 0.0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        total += F.cross_entropy(model(xb), yb, reduction="sum").item()
        n += yb.numel()
    model.train()
    return total / n

### 조기 종료와 체크포인트

**valid 손실이 최저인 순간의 가중치를 따로 저장**해 두는 게 핵심입니다.
학습이 더 진행되어 모델이 나빠져도, 최고 시점으로 되돌릴 수 있습니다.

In [ ]:
import copy

def train(model, epochs=60, lr=3e-3, patience=10, log_every=10):
    model = model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    best_val, best_state, best_epoch, bad = float("inf"), None, 0, 0
    history = []

    for ep in range(1, epochs + 1):
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            loss = F.cross_entropy(model(xb), yb)
            opt.zero_grad(); loss.backward(); opt.step()

        tr = eval_loss(model, train_loader)
        va = eval_loss(model, val_loader)
        history.append((ep, tr, va))

        if va < best_val:
            best_val, best_epoch, bad = va, ep, 0
            best_state = copy.deepcopy(model.state_dict())   # 체크포인트
        else:
            bad += 1

        if ep % log_every == 0 or ep == 1:
            flag = " *best*" if ep == best_epoch else ""
            print(f"epoch {ep:3d} | train {tr:6.3f} (ppl {math.exp(tr):8.2f})"
                  f" | valid {va:6.3f} (ppl {math.exp(va):8.2f}){flag}")

        if bad >= patience:
            print(f"{NEWLINE}valid 손실이 {patience} 에폭 동안 개선되지 않아 조기 종료합니다.")
            break

    model.load_state_dict(best_state)      # 최고 시점으로 되돌리기
    print(f"{NEWLINE}최고 성능: epoch {best_epoch}, valid loss {best_val:.3f} (ppl {math.exp(best_val):.2f})")
    return model, history

NEWLINE = chr(10)
set_seed(1234)
model = NPLM(V)
print(f"파라미터 {human(count_params(model))},  기준선 ppl = V = {V}{NEWLINE}")
model, hist = train(model)

### 학습 곡선 보기

숫자만 보면 감이 안 옵니다. 그림으로 그리면 과적합 지점이 한눈에 보입니다.

In [ ]:
import matplotlib.pyplot as plt

eps = [h[0] for h in hist]
plt.figure(figsize=(7, 4))
plt.plot(eps, [h[1] for h in hist], label="train loss")
plt.plot(eps, [h[2] for h in hist], label="valid loss")
best_ep = min(hist, key=lambda h: h[2])[0]
plt.axvline(best_ep, ls="--", c="gray", label=f"best (epoch {best_ep})")
plt.axhline(math.log(V), ls=":", c="red", label=f"baseline log(V)={math.log(V):.2f}")
plt.xlabel("epoch"); plt.ylabel("cross entropy loss"); plt.legend(); plt.grid(alpha=0.3)
plt.title("train vs valid")
plt.tight_layout(); plt.show()

**이 그림이 딥러닝에서 가장 많이 보게 될 그림입니다.**

- 두 곡선이 같이 내려감 → 건강한 학습
- train만 내려가고 valid가 올라감 → **과적합 시작**. 두 선 사이의 간격이 과적합의 크기
- 둘 다 안 내려감 → 학습률이 잘못됐거나 모델이 너무 작음

### 방금 무슨 일이 일어났는지 정확히 봅시다

로그를 보면 **최고 성능이 epoch 1** 입니다. 그리고 조기 종료가 걸렸습니다.

무슨 뜻일까요? **valid 손실이 단 한 번도 개선되지 않았다**는 뜻입니다.
모델은 첫 에폭부터 곧장 외우기 시작했고, 일반화는 전혀 일어나지 않았습니다.
조기 종료가 정직하게 작동해서 "거의 학습되지 않은 epoch 1 가중치"를 돌려준 겁니다.

> **이게 정상입니다.** 학습 샘플 351개, 파라미터 70,440개.
> 이 조건에서 일반화를 기대하는 건 무리입니다.
> 조기 종료는 성능을 만들어 내는 장치가 아니라 **더 나빠지는 걸 막는 장치**입니다.

### 그래서 생성 실험용 모델을 따로 만듭니다

지금 `model` 은 사실상 학습이 안 된 상태라 글을 쓰게 해 봐야 무작위 단어 나열입니다.
뒤에서 할 일은 **생성 전략(그리디/온도/top-k)이 무엇을 바꾸는지 관찰하는 것**이므로,
일부러 **학습 데이터에 과적합시킨 모델**을 따로 하나 만들어 쓰겠습니다.

과적합 모델은 "학습 문장을 외운 모델"입니다. 생성 결과가 학습 문장을 베낄 텐데,
**그 베끼는 모습 자체가 5절에서 진단할 대상**입니다.

In [ ]:
# 생성 실험용: 드롭아웃 없이, 조기 종료 없이, 일부러 과적합시킵니다.
set_seed(1234)
gen_model = NPLM(V, dropout=0.0).to(device)
opt = torch.optim.Adam(gen_model.parameters(), lr=3e-3)
for ep in range(60):
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        loss = F.cross_entropy(gen_model(xb), yb)
        opt.zero_grad(); loss.backward(); opt.step()

tr, va = eval_loss(gen_model, train_loader), eval_loss(gen_model, val_loader)
print(f"과적합 모델   train {tr:.3f} (ppl {math.exp(tr):7.2f})  |  valid {va:.3f} (ppl {math.exp(va):10.2f})")
print("train 혼란도는 1에 가깝고 valid는 폭발합니다 = 학습 문장을 완벽히 외운 상태.")

## 3. CPU vs MPS — 언제 GPU가 이기는가

M1 맥에서 `mps` 를 쓰면 무조건 빨라질 거라 생각하기 쉽지만 **그렇지 않습니다.**
GPU는 데이터를 옮기는 비용이 있어서, **연산량이 충분히 커야** 이깁니다.

In [ ]:
def bench(dev, batch, d=64, hidden=128, iters=50):
    set_seed(0)
    m = NPLM(V, d=d, hidden=hidden).to(dev)
    o = torch.optim.Adam(m.parameters(), lr=1e-3)
    x = torch.randint(0, V, (batch, CTX), device=dev)
    y = torch.randint(0, V, (batch,), device=dev)
    for _ in range(5):                       # 워밍업
        l = F.cross_entropy(m(x), y); o.zero_grad(); l.backward(); o.step()
    if dev.type == "mps":
        torch.mps.synchronize()
    t0 = time.perf_counter()
    for _ in range(iters):
        l = F.cross_entropy(m(x), y); o.zero_grad(); l.backward(); o.step()
    if dev.type == "mps":
        torch.mps.synchronize()              # GPU는 비동기 -- 기다려야 정확히 잽니다
    return (time.perf_counter() - t0) / iters * 1000

cpu = torch.device("cpu")
print(f"{'배치 크기':>8} | {'CPU (ms)':>10} | {'MPS (ms)':>10} | {'승자':>8}")
print("-" * 46)
for batch in (32, 256, 2048, 8192):
    tc = bench(cpu, batch)
    tm = bench(device, batch) if device.type != "cpu" else float("nan")
    win = "-" if device.type == "cpu" else ("MPS" if tm < tc else "CPU")
    print(f"{batch:>8} | {tc:>10.2f} | {tm:>10.2f} | {win:>8}")

### 실무 교훈

**배치가 작을 땐 CPU가 이깁니다.** 커널 실행 오버헤드가 연산 시간보다 크기 때문입니다.
배치를 키울수록 GPU가 유리해집니다.

이것이 GPU 실습에서 **배치 크기를 최대한 키우라**고 하는 이유입니다.
"GPU 메모리가 허락하는 한 가장 큰 배치"가 기본 전략입니다.

> **M1 8GB 주의**: `mps` 는 시스템 메모리를 공유합니다.
> 배치를 너무 키우면 스왑이 발생해 오히려 급격히 느려지거나 죽습니다.
> 적당한 값을 찾은 뒤 `gradient accumulation` 으로 유효 배치를 키우는 방법이 있습니다.

### 그 밖의 최적화 감각

| 기법 | 효과 | 주의 |
|---|---|---|
| 배치 키우기 | 처리량↑ | 메모리, 일반화 저하 가능 |
| `torch.compile` | 커널 융합 | MPS 지원이 제한적 |
| AMP (fp16/bf16) | 메모리↓ 속도↑ | CUDA에서 효과가 큼 |
| `num_workers` | 데이터 로딩 병렬화 | 작은 데이터엔 오버헤드 |
| gradient accumulation | 유효 배치↑ | 스텝 수 늘어남 |

## 4. 생성 — 모델에게 글을 쓰게 하기

학습된 모델이 진짜로 무엇을 배웠는지 보는 가장 직관적인 방법은 **글을 쓰게 해 보는 것**입니다.

### 생성의 원리

1. 문맥을 넣어 다음 단어의 확률 분포를 얻는다
2. 그 분포에서 단어 하나를 고른다
3. 고른 단어를 문맥 뒤에 붙이고 1로 돌아간다

**2번 "어떻게 고르느냐"가 생성 품질을 좌우합니다.**

In [ ]:
@torch.no_grad()
def generate(model, prompt=None, max_new=15, strategy="greedy", temperature=1.0, top_k=10):
    model.eval()
    ctx = [BOS] * CTX
    if prompt:
        ctx = (ctx + tokenize(prompt))[-CTX:]
    out = list(prompt.split()) if prompt else []

    for _ in range(max_new):
        ids = torch.tensor([[stoi.get(w, stoi[UNK]) for w in ctx]]).to(device)
        logits = model(ids)[0]

        if strategy == "greedy":
            nxt = logits.argmax().item()
        else:
            logits = logits / max(temperature, 1e-6)
            if strategy == "top_k":
                v, _ = logits.topk(min(top_k, logits.size(-1)))
                logits = logits.masked_fill(logits < v[-1], float("-inf"))
            nxt = torch.multinomial(F.softmax(logits, dim=-1), 1).item()

        word = itos[nxt]
        out.append(word)
        ctx = (ctx + [word])[-CTX:]
        if word == ".":
            break
    model.train()
    return " ".join(out)

### (a) 그리디 — 항상 가장 확률 높은 단어

In [ ]:
for prompt in ["언어 모델은", "트랜스포머는", "사전 훈련은"]:
    print(f"[{prompt}] -> {generate(gen_model, prompt, strategy='greedy')}")

print(f"{chr(10)}같은 프롬프트를 3번 반복하면:")
for _ in range(3):
    print("  ", generate(gen_model, "언어 모델은", strategy="greedy"))

**그리디는 항상 똑같은 답을 냅니다.** 무작위성이 전혀 없으니까요.

장점은 안정적이라는 것, 단점은 **지루하고 반복에 빠지기 쉽다**는 것.
실제로 그리디 생성은 `그리고 그리고 그리고...` 같은 루프에 자주 갇힙니다.

### (b) 온도 샘플링 — 확률대로 뽑되, 분포를 날카롭게/뭉뚝하게

$$P_i = \frac{\exp(z_i / T)}{\sum_j \exp(z_j / T)}$$

- $T \to 0$: 가장 큰 값만 남음 = 그리디
- $T = 1$: 모델이 계산한 확률 그대로
- $T > 1$: 분포가 평평해짐 = 과감해짐

먼저 온도가 분포를 어떻게 바꾸는지 눈으로 봅시다.

In [ ]:
logits_demo = torch.tensor([3.0, 2.0, 1.0, 0.5, 0.0])
print(f"{'온도':>6} | " + " | ".join(f"{i:>6}" for i in range(5)))
print("-" * 52)
for T in (0.1, 0.5, 1.0, 1.5, 3.0):
    p = F.softmax(logits_demo / T, dim=-1)
    print(f"{T:>6.1f} | " + " | ".join(f"{v:6.3f}" for v in p))
print(f"{chr(10)}낮은 온도 -> 1등에 몰빵 / 높은 온도 -> 골고루")

In [ ]:
set_seed(7)
for T in (0.3, 1.0, 2.0, 3.0):
    outs = [generate(gen_model, "언어 모델은", strategy="sample", temperature=T) for _ in range(4)]
    print(f"--- 온도 {T}  (서로 다른 결과 {len(set(outs))}/4) ---")
    for o in outs:
        print("  ", o)

### 온도 결과에서 배울 것

**낮은 온도(0.3, 1.0)에서는 4번 모두 같은 문장이 나옵니다.** 온도를 바꿨는데 왜 안 변할까요?

이 모델은 학습 문장을 외워서 다음 단어에 거의 1.0의 확률을 주고 있습니다.
분포가 이미 극단적으로 뾰족해서, 온도로 나눠도 1등이 여전히 1등입니다.

> **온도는 분포의 모양을 바꿀 뿐, 없는 다양성을 만들어 내지 못합니다.**
> 확신에 찬 모델은 온도를 올려도 잘 안 흔들리고, 충분히 올리면 그제야 **무너집니다**
> — 그럴듯한 대안이 아니라 아무 단어나 나오는 식으로요. 온도 3.0 결과를 보세요.

실제 LLM에서 온도가 잘 듣는 이유는, 그 모델들이 **여러 그럴듯한 후보에 확률을 나눠 갖고**
있기 때문입니다. 나눠 가질 만큼 세상을 많이 봤다는 뜻이죠.

### (c) Top-k 샘플링 — 후보를 상위 k개로 제한

온도만 올리면 **꼬리에 있는 엉뚱한 단어**가 튀어나옵니다.
확률이 0.0001이어도 뽑힐 수는 있으니까요. 한 번 어긋나면 그 뒤가 전부 무너집니다.

Top-k는 상위 k개만 남기고 나머지를 잘라냅니다.
온도를 3.0으로 높게 두고 k만 바꿔 보면 효과가 분명히 보입니다.

In [ ]:
set_seed(7)
for k in (1, 5, 50):
    outs = [generate(gen_model, "언어 모델은", strategy="top_k", temperature=3.0, top_k=k) for _ in range(3)]
    print(f"--- top_k={k}, 온도 3.0 (일부러 높게) ---")
    for o in outs:
        print("  ", o)

> **top_k=1 은 그리디와 같습니다.** 후보가 1개면 고를 게 없으니까요.
> 온도가 3.0이어도 k=1이면 결과가 흔들리지 않습니다 — **top-k가 온도를 제어**한 것입니다.
> 이것이 실무에서 온도와 top-k/top-p를 **함께** 쓰는 이유입니다.

실제 API에서 보는 파라미터들이 전부 이 이야기입니다.

| 파라미터 | 하는 일 | 올리면 |
|---|---|---|
| `temperature` | 분포의 날카로움 | 창의적/불안정 |
| `top_k` | 후보 개수 제한 | 다양해짐 |
| `top_p` (nucleus) | 누적 확률 p까지만 후보 | 다양해짐 |

**작업에 따라 다르게 씁니다**: 코드 생성·사실 질의응답은 낮은 온도(0~0.3),
창작·브레인스토밍은 높은 온도(0.8~1.2).

## 5. 생성 결과를 어떻게 분석할 것인가

생성문이 그럴듯해 보여도 **모델이 실제로 무엇을 하고 있는지는 따로 확인해야 합니다.**
두 가지를 재 봅시다: (1) 학습 문장을 베끼는가, (2) 얼마나 확신하는가.

In [ ]:
# 진단 1: 생성문이 학습 문장을 그대로 베낀 것인가?
train_set = set(train_sents)
set_seed(0)
gens = [generate(gen_model, "언어 모델은", strategy="sample", temperature=0.8) for _ in range(20)]

copied = sum(1 for g in gens if any(g.replace(" .", ".") in t or t in g for t in train_set))
print(f"학습 문장을 통째로 베낀 비율: {copied}/{len(gens)}")
print(f"서로 다른 생성문 개수      : {len(set(gens))}/{len(gens)}")
print()
for g in gens[:5]:
    print("  ", g)

In [ ]:
# 진단 2: 모델은 얼마나 확신하고 있는가? (다음 단어 분포의 엔트로피)
@torch.no_grad()
def entropy_at(m, prompt):
    ctx = ([BOS] * CTX + tokenize(prompt))[-CTX:]
    ids = torch.tensor([[stoi.get(w, stoi[UNK]) for w in ctx]]).to(device)
    p = F.softmax(m(ids)[0], dim=-1)
    ent = -(p * (p + 1e-12).log()).sum().item()
    top = p.topk(3)
    return ent, [(itos[i.item()], round(v.item(), 3)) for v, i in zip(top.values, top.indices)]

print(f"최대 엔트로피(완전 무지) = log(V) = {math.log(V):.3f}{chr(10)}")
for m, label in ((gen_model, "과적합 모델"), (model, "조기 종료된 모델")):
    m.eval()
    print(f"=== {label} ===")
    for prompt in ["언어 모델은", "트랜스포머는", "완전히 엉뚱한 문맥"]:
        ent, top = entropy_at(m, prompt)
        print(f"  [{prompt:>14}]  엔트로피 {ent:.3f}   상위후보 {top}")
    m.train()
    print()

### 진단 읽는 법

- **엔트로피가 낮다** = 모델이 확신함. 학습 문장을 외운 문맥이면 거의 0에 가깝습니다
- **엔트로피가 $\log V$ 에 가깝다** = 아무것도 모름
- **생성문이 학습 문장의 복사본** = 과적합의 결정적 증거

두 모델을 나란히 놓고 보면 대비가 분명합니다.

- **과적합 모델**: 외운 문맥에서 엔트로피가 낮습니다. 확신에 차서 외운 다음 단어를 뱉습니다.
  그런데 그 확신은 "이해"가 아니라 "암기"입니다 — 처음 보는 문맥에서도 근거 없이 확신합니다.
- **조기 종료된 모델**: 엔트로피가 $\log V$ 근처. 아직 아무것도 모르는 상태입니다.

**둘 다 좋은 모델이 아닙니다.** 좋은 모델은 아는 것에 확신하고 모르는 것에 겸손합니다.
그 상태에 도달하려면 데이터가 훨씬 많아야 합니다.

### 그래서 무엇이 더 필요한가

| 지금 | 필요한 것 |
|---|---|
| 50문장 | 수백만 문장 |
| 문맥 3단어 고정 | 수백~수천 토큰 가변 문맥 |
| 위치별 가중치 분리 | 위치와 무관한 가중치 공유 |
| 은닉층 1개 | 깊은 블록 반복 |

1번은 **데이터**, 2~4번은 **구조**의 문제입니다.
구조 문제를 푸는 것이 다음 파트의 주제입니다.

---

## 정리

**학습 루프 체크리스트** (앞으로 모든 학습에 적용하세요)

- [ ] 학습/평가 데이터를 나눴는가
- [ ] 미니배치 + 셔플을 쓰는가
- [ ] valid 손실을 매 에폭 재는가
- [ ] 최저 valid 시점의 가중치를 저장하는가
- [ ] 학습 곡선을 그려 봤는가
- [ ] 기준선(log V)과 비교했는가

**생성 전략**

| 전략 | 무작위성 | 쓰임새 |
|---|---|---|
| greedy | 없음 | 재현성이 필요할 때 |
| temperature | 조절 가능 | 범용 |
| top-k / top-p | 꼬리 차단 | 실무 기본값 |

## 연습

1. `dropout` 을 0.0 / 0.5 로 바꿔 학습 곡선이 어떻게 달라지는지 비교하세요.
2. 큰 한국어 말뭉치(4-2 참고)로 이 노트북을 통째로 다시 돌려 보세요.
   생성문이 "외운 문장"에서 "새로 만든 문장"으로 바뀌는 순간이 옵니다.
3. `top_p`(nucleus) 샘플링을 직접 구현해 `generate` 에 추가해 보세요.

## 다음

→ `../../Part3/Ch5/0501_Attention.ipynb` : 고정 문맥에서 탈출하는 장치, 어텐션